# Graph Attention Networks (2-16)

## Introduction
* Basic idea: in a graph structure some neighbors are more important and some connections are weak but basic GNN structures average all neighbors and connections equally.
* Uses an attention mechanism to learn which neighbors are more important instead of averaging the neighbors equally like a graph convolutional network.

## Mechanism
### Overview
During the message passing stage:
1. Each node looks at its neighbors
2. Computes an importance score for each neighbor
3. Weights neighbors accordingly while aggregating all of the information
### Details
Let each node have a set of features $h_i$ and a set of neighbors $N_i$.
1. The features of each node get multiplied by a learnable weights matrix: $z_i = Wh_i$
2. Compute the attention score between the current node's features, $z_i$ and all neighboring nodes' features $z_j$. The attention score measure how import node j is to node i. The attenion score is then $e_{ij} = LeakyRelu(a^T[z_i||z_j])$ where:
    * a is a learnable attention vector
    * $[||]$ is a concatenation (i.e. if $z_i$, $z_j$ are of length F, then $[z_i||z_j]$ is of length 2F)
3. Normalize the attention weights using the softmax function so the sum of attention weights over all neighbors adds to 1. 100% of the attention is distributed over all neighbors.
4. Compute the output of each node using an aggregation based on the attention weights. In the below equation $h_i^\prime$ is the output of the message passing on the node, $\sigma$ is a nonlinear activation function, $\alpha_{ij}$ represent the normalized attentions scores, and $z_j$ are the features (multiplied by the weights) from the neighboring nodes.
$$h_i^\prime = \sigma(\sum_{j\in N_i}\alpha_{ij}z_j)$$


## Multi-Headed Attention
Instead of computing one set of $\alpha_{ij}$ values per node, using a learnable attention vector, many implementaitons will compute K sets of attention scores using K learnable attention vectors. The final output of node i, $h_i^\prime$ is then the average over the outputs using the K separate attention scores.

## Differences from Transformers
* No global connections in the attention mechanism
* No positional encoding or sequence order.

## Example
Consider an example where you are attempting to determine if a person is likely to buy a product based on their social network (because you don't want to spend advertising funds if someone is unlikely to buy the product). A close friend who has bought a similar product will recieve a high weight, an aquaintance who has bought a similar product will recieve a low weight, and a random connection who has bought a similar product will recieve almost no weight.

## Limitations
* Attention computations are expensive so not feasible for large networks, GCN may be bettter for large networks even with the equal weighting
* Still only considers direct neighbors in the first message passing stage. Long range connections require stacks of layers.
* Can be overfit on small graphs.


## PyTorch Implementation

In [ ]:
#############
## IMPORTS ##
#############
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_dense_adj

In [10]:
###################################
## GRAPH ATTENTION NETWORK LAYER ##
###################################
# Create a single graph attention layer based on the original GAT paper (https://arxiv.org/abs/1710.10903).
# Multiple attention heads can be implemented by stacking multiple GAT layers and concatenating their outputs.

class GATLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.6, alpha=0.2):
        """
        Inputs:
            in_features: Number of input features per node
            out_features: Number of output features per node
            dropout: Dropout rate for attention coefficients
            alpha: Negative slope for LeakyReLU
        Outputs:
            None.
        Initializes the learnable parameters for the GAT layer, including the weight matrix and attention mechanism.

        """
        # Call the parent class constructor
        super(GATLayer, self).__init__()
        # Initialize the weight matrix for feature transformation. Here, we set bias=False because the original GAT paper 
        # does not use a bias term in the linear transformation, but you can set it to True if you want to include a bias.
        # Passing the input through a linear layer without a bias or a activiation function simply applies a weights matrix
        # to the input.
        self.W = nn.Linear(in_features, out_features, bias=False)

        # Attention mechanism. The attention vector is made learnable by making it the output of a single neural network layer. 
        # The input to this layer is the concatenation of the transformed features of two nodes (h_i and h_j), which results in a 
        # vector of size 2 * out_features. The output is a single scalar that represents the attention coefficient for the edge between 
        # those two nodes.
        self.a = nn.Linear(2 * out_features, 1, bias=False)

        # Define the activation function for the attention mechanism and the dropout layer for regularization. The LeakyReLU is used to
        # allow for negative attention coefficients, which can help with learning. The dropout is applied to the attention coefficients to
        # prevent overfitting.
        self.leakyrelu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h, adj):
        """
        Inputs:
            h: Node features (N, in_features)
            adj: Adjacency matrix (N, N)
        Outputs:
            h_prime: Updated node features (N, out_features)
        The forward method computes the output of the GAT layer. It first applies a linear transformation to the input features, then computes 
        the attention coefficients for each edge based on the transformed features. The attention coefficients are masked to only consider
        the edges present in the adjacency matrix. Finally, the output features for each node are computed as a weighted sum of the transformed 
        features of its neighbors, where the weights are the attention coefficients.
        """
        # Apply the weight matrix to the input features and get the size of the transformed features.
        z = self.W(h)  # (N, out_features)
        N = z.size(0)

        # Prepare attention input using the equation earlier in this notebook. We need to compute the attention coefficients for each pair of nodes (i, j). 
        # To do this, we create a new tensor that contains the concatenated features of all pairs of nodes.
        a_input = torch.cat([
            z.repeat(1, N).view(N * N, -1),
            z.repeat(N, 1)
        ], dim=1).view(N, N, 2 * z.size(1))
        # Apply the leaky relu activation function to the output of the attention mechanism to get the attention coefficients. The squeeze(2) is used to 
        # remove the last dimension, since the output of the attention mechanism is a single scalar for each pair of nodes.
        e = self.leakyrelu(self.a(a_input).squeeze(2))

        # Mask non-edges. Note that the attention coefficients for non-existent edges are set to a very large negative value (effectively -infinity) so that 
        # after applying the softmax, they will be zero. This ensures that only the neighbors of each node contribute to the final output.
        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)

        # Apply softmax to get the attention coefficients and then apply dropout for regularization. The softmax is applied along the dimension of the neighbors 
        # (dim=1) so that the attention coefficients for each node sum to 1.
        attention = F.softmax(attention, dim=1)
        attention = self.dropout(attention)

        # Compute the final output features for each node as a weighted sum of the transformed features of its neighbors, where the weights are the attention 
        # coefficients.
        h_prime = torch.matmul(attention, z)

        # Apply an activation function to the output features. In the original GAT paper, they use an ELU activation function, but you can choose any activation 
        # function you like.
        return F.elu(h_prime)


In [ ]:
####################################
## GRAPH ATTENTION NETWORK MODEL ##
####################################

# Create a GAT model that consists of multiple GAT layers. The first layer will have multiple attention heads, and the output of these heads
#  will be concatenated. The final layer will be a single GAT layer that outputs the class probabilities for each node.   

class GAT(nn.Module):
    def __init__(self, n_features, n_hidden, n_classes, n_heads=8, dropout=0.6):
        """
        Inputs:
            n_features: Number of input features per node
            n_hidden: Number of hidden units in each attention head
            n_classes: Number of output classes for node classification
            n_heads: Number of attention heads in the first layer
            dropout: Dropout rate for regularization
        Outputs:
            None.
        Initializes the GAT model with multiple attention heads in the first layer and a final output layer that produces class 
        probabilities for each node. The first layer consists of multiple GAT layers (one for each attention head), and their outputs 
        are concatenated.
        """
        # Call the parent class constructor
        super(GAT, self).__init__()
        # Define the dropout layer for regularization. Dropout is applied to the input features and the attention coefficients to 
        # prevent overfitting.
        self.dropout = nn.Dropout(dropout)

        # Multi-head attention. The first layer consists of multiple GAT layers, each representing an attention head. The output of each 
        # head is concatenated to form the input for the next layer.
        self.attentions = nn.ModuleList([
            GATLayer(n_features, n_hidden, dropout=dropout)
            for _ in range(n_heads)
        ])

        # Output layer
        self.out_att = GATLayer(n_hidden * n_heads, n_classes, dropout=dropout)

    def forward(self, x, adj):
        """
        Inputs:
            x: Node features (N, n_features)
            adj: Adjacency matrix (N, N)
        Outputs:
            Node class probabilities (N, n_classes)
        The forward method computes the output of the GAT model. It first applies dropout to the input features, then computes the
        multi-head attention outputs for the first layer, concatenates them, applies dropout again, and finally computes the output
        class probabilities using the final GAT layer.
        """
        # Apply dropout to the input features for regularization. This helps prevent overfitting by randomly setting some of the input 
        # features to zero during training.
        x = self.dropout(x)

        # Concatenate multi-head outputs
        x = torch.cat([att(x, adj) for att in self.attentions], dim=1)

        # Apply dropout again before the final output layer for regularization. This helps prevent overfitting by randomly setting some 
        # of the features to zero
        x = self.dropout(x)
        x = self.out_att(x, adj)

        # Apply a log softmax to the output to get the class probabilities for each node. The log softmax is used instead of softmax because 
        # it is more numerically stable when computing the loss.
        return F.log_softmax(x, dim=1)


In [11]:
########################
## IMPORT THE DATASET ##
########################

# Load Cora dataset
dataset = Planetoid(root='./data', name='Cora')
data = dataset[0]

# Convert sparse edge_index to dense adjacency. This is necessary because the GAT layer we implemented expects a dense adjacency matrix. 
# The to_dense_adj function converts the sparse edge_index format used by PyTorch Geometric into a dense adjacency matrix. We also add 
# self-loops to the adjacency matrix by adding an identity matrix, which allows each node to attend to itself in the attention mechanism.
adj = to_dense_adj(data.edge_index)[0]
adj = adj + torch.eye(adj.size(0))  # add self-loops

# Prepare the features, labels, and masks for training, validation, and testing. The features are the input node features, and the labels are 
# the target class labels for each node.
features = data.x
labels = data.y
train_mask = data.train_mask
val_mask = data.val_mask
test_mask = data.test_mask

In [ ]:
######################
## DEVICE SELECTION ##
######################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the features, labels, and adjacency matrix to the selected device (GPU or CPU) for training. 
# This allows us to take advantage of GPU acceleration if available.
features = features.to(device)
labels = labels.to(device)
adj = adj.to(device)



In [12]:
######################
## DEFINE THE MODEL ##
######################

# Define a graph attention network model with the specified number of input features, hidden units, output classes, and attention heads. 
# The model is moved to the selected device (GPU or CPU) for training. Here we are using 8 hidden units for each attention head and 2 attention 
# heads in the first layer, which results in a total of 16 hidden units after concatenation. This can be adjusted based on the complexity of the 
# dataset and the computational resources available. Currently we are using a small number of hidden units and attention heads for simplicity.
model = GAT(
    n_features=features.shape[1],
    n_hidden=8,
    n_classes=dataset.num_classes,
    n_heads=2
).to(device)

# Define the optimizer for training the model. The learning rate is set to 0.005, and a weight decay of 5e-4 is applied for regularization to prevent overfitting.
# These can be adjusted as needed.
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

In [ ]:
###############################################
## DEFINE THE TRAINING AND TESTING FUNCTIONS ##
###############################################
def train():
    """
    Inputs:
        None.
    Outputs:
        loss.item(): The loss value for the current training step.
    The train function performs a single training step for the GAT model. It sets the model to training mode, zeroes the gradients, 
    computes the output of the model, calculates the loss using the negative log-likelihood loss function, performs backpropagation, 
    and updates the model parameters using the optimizer. The loss value is returned for monitoring the training process.
    """
    model.train()
    optimizer.zero_grad()
    output = model(features, adj)
    loss = F.nll_loss(output[train_mask], labels[train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


def test(mask):
    """
    Inputs:
        mask: A boolean mask indicating which nodes to evaluate.
    Outputs:
        accuracy: The accuracy of the model on the specified nodes.
    The test function evaluates the model's performance on a given set of nodes. It sets the model to evaluation mode, 
    computes the output of the model without computing gradients, and calculates the accuracy by comparing the predicted 
    labels with the true labels for the specified nodes.
    """
    model.eval()
    with torch.no_grad():
        output = model(features, adj)
        pred = output.argmax(dim=1)
        correct = pred[mask].eq(labels[mask]).sum().item()
        return correct / mask.sum().item()




In [13]:
###################
## TRAINING LOOP ##
###################
for epoch in range(50):
    loss = train()
    if epoch % 5 == 0:
        val_acc = test(val_mask)
        print(f"Epoch {epoch}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f}")

test_acc = test(test_mask)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")

Epoch 0, Loss: 1.9531, Val Acc: 0.4240
Epoch 5, Loss: 1.8518, Val Acc: 0.6800
Epoch 10, Loss: 1.7069, Val Acc: 0.7140
Epoch 15, Loss: 1.5582, Val Acc: 0.7460
Epoch 20, Loss: 1.5285, Val Acc: 0.7720
Epoch 25, Loss: 1.3260, Val Acc: 0.7840
Epoch 30, Loss: 1.2391, Val Acc: 0.7820
Epoch 35, Loss: 1.1064, Val Acc: 0.7900
Epoch 40, Loss: 1.0551, Val Acc: 0.7940
Epoch 45, Loss: 0.8505, Val Acc: 0.7980

Final Test Accuracy: 0.8080


## Resources
* [Graph Attention Networks (Original Paper)](https://arxiv.org/abs/1710.10903)
* [Graph Attention Networks](https://medium.com/@ashish28082002.ak/graph-attention-networks-52f03591b3cc)
* [Graph Attention Networks](https://petar-v.com/GAT/)
    * Blog post by one of the authors of the original paper
* [Understand Graph Attention Network](https://www.dgl.ai/dgl_docs/en/2.0.x/tutorials/models/1_gnn/9_gat.html)
* [Graph Attention Networks Paper Explained With Illustration and PyTorch Implementation](https://epichka.com/blog/2023/gat-paper-explained/)
* [Graph Attention Networks (GAT)](https://nn.labml.ai/graphs/gat/index.html)
    * General recommendation for [Annotated Research Papers](https://nn.labml.ai), the base website of the above paper. It has implementations of algorothms from famous ML and AI papers.